Cutting the data 

In [3]:
import os
import pandas as pd

# Define the input directory
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'

# Define the cutoff date (April 1, 2020)
cutoff_date = pd.to_datetime('2020-04-01')

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)
            
            # Check if 'datetime' column exists
            if 'datetime' in df.columns:
                # Convert the 'datetime' column to datetime type
                df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
                
                # Filter the data to start from April 1, 2020
                df = df[df['datetime'] >= cutoff_date]
            
            # Save the edited data back to the same file
            df.to_csv(file_path, index=False)
            print(f"Updated file saved: {filename}")
        
        except Exception as e:
            print(f"Error processing file {filename}: {e}")

print("Processing complete.")


Updated file saved: EARLWOOD_AQMS_Processed.csv
Updated file saved: SINGLETON_AQMS_Processed.csv
Updated file saved: MAYFIELD_AQMS_Processed.csv
Updated file saved: JERRYS-PLAINS_AQMS_Processed.csv
Updated file saved: MERRIWA_AQMS_Processed.csv
Updated file saved: COOK-AND-PHILLIP_AQMS_Processed.csv
Updated file saved: GUNNEDAH_AQMS_Processed.csv
Updated file saved: CAMDEN_AQMS_Processed.csv
Updated file saved: BATHURST_AQMS_Processed.csv
Updated file saved: PORT-MACQUARIE_AQMS_Processed.csv
Updated file saved: NARRABRI_AQMS_Processed.csv
Updated file saved: CARRINGTON_AQMS_Processed.csv
Updated file saved: CAMBERWELL_AQMS_Processed.csv
Updated file saved: COFFS-HARBOUR_AQMS_Processed.csv
Updated file saved: MUSWELLBROOK-NW_AQMS_Processed.csv
Updated file saved: BRADFIELD-HIGHWAY_AQMS_Processed.csv
Updated file saved: ALEXANDRIA_AQMS_Processed.csv
Updated file saved: GOULBURN_AQMS_Processed.csv
Updated file saved: WOLLONGONG_AQMS_Processed.csv
Updated file saved: BRINGELLY_AQMS_Process

Missing Value Report

In [4]:

import os
import pandas as pd

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'
output_file = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/missing_value_report_01.csv'

# List to store missing value data
missing_data = []

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)
            
            # Calculate missing value percentage for each column
            missing_percentages = df.isnull().mean() * 100
            
            # Store the result for each column
            for column, missing_percentage in missing_percentages.items():
                missing_data.append({
                    'study_site': study_site,
                    'column': column,
                    'missing_percentage': missing_percentage
                })
        
        except Exception as e:
            print(f"Error processing file {filename}: {e}")

# Convert the results into a DataFrame
missing_df = pd.DataFrame(missing_data)

# Save the missing values report to a CSV file
missing_df.to_csv(output_file, index=False)

print(f"Missing value report saved to {output_file}")


Missing value report saved to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/missing_value_report_01.csv


In [1]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Drop the 'WGU' column if it exists
            if 'WGU' in df.columns:
                df = df.drop(columns=['WGU'])
                print(f"Dropped 'WGU' column from {filename}")

            # Drop columns that have more than 50% missing values
            missing_threshold = 0.5
            df = df.dropna(thresh=int(len(df) * (1 - missing_threshold)), axis=1)
            print(f"Dropped columns with more than 50% missing values from {filename}")

            # Drop columns that have all missing values
            df = df.dropna(axis=1, how='all')

            # Sort the data by 'Time' column, keeping the 'Date' and 'Time' unchanged
            df = df.sort_values(by='Time')

            # Select only the numerical columns for imputation (excluding 'Date', 'Time', etc.)
            numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

            # Apply KNN imputation only to the numerical columns
            if not numerical_columns.empty:
                df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

            # Save the imputed data to a new CSV file in the output directory
            output_file = os.path.join(output_dir, f'{study_site}_imputed.csv')
            df.to_csv(output_file, index=False)
            print(f"Processed and saved imputed data for {study_site} to {output_file}")

        except Exception as e:
            print(f"Error processing file {filename}: {e}")


Dropped 'WGU' column from EARLWOOD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from EARLWOOD_AQMS_Processed.csv
Processed and saved imputed data for EARLWOOD to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/EARLWOOD_imputed.csv
Dropped 'WGU' column from SINGLETON_AQMS_Processed.csv
Dropped columns with more than 50% missing values from SINGLETON_AQMS_Processed.csv
Processed and saved imputed data for SINGLETON to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/SINGLETON_imputed.csv
Dropped 'WGU' column from MAYFIELD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from MAYFIELD_AQMS_Processed.csv
Processed and saved imputed data for MAYFIELD to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/MAYFIELD_imputed.csv
Dropped 'WGU' column from JERRYS-PLAINS_AQMS_Processed.csv
Dropped columns with more than 50% missing values from JERRYS-PLAINS_AQMS_Proce

In [2]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL2'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Function to fill in missing timestamps
def ensure_continuous_datetime(df, date_col, time_col):
    # Combine date and time into a single datetime column
    df['datetime'] = pd.to_datetime(df[date_col] + ' ' + df[time_col])

    # Create a complete range of datetimes
    full_datetime_range = pd.date_range(start=df['datetime'].min(), end=df['datetime'].max(), freq='H')

    # Reindex the dataframe to include all timestamps
    df = df.set_index('datetime').reindex(full_datetime_range).reset_index().rename(columns={'index': 'datetime'})

    # Split back into separate 'Date' and 'Time' columns
    df[date_col] = df['datetime'].dt.date
    df[time_col] = df['datetime'].dt.time

    # Drop the combined datetime column
    df = df.drop(columns=['datetime'])

    return df

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Drop the 'WGU' column if it exists
            if 'WGU' in df.columns:
                df = df.drop(columns=['WGU'])
                print(f"Dropped 'WGU' column from {filename}")

            # Drop columns that have more than 50% missing values
            missing_threshold = 0.5
            df = df.dropna(thresh=int(len(df) * (1 - missing_threshold)), axis=1)
            print(f"Dropped columns with more than 50% missing values from {filename}")

            # Drop columns that have all missing values
            df = df.dropna(axis=1, how='all')

            # Ensure continuous date and time
            df = ensure_continuous_datetime(df, date_col='Date', time_col='Time')

            # Select only the numerical columns for imputation (excluding 'Date', 'Time', etc.)
            numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

            # Apply KNN imputation only to the numerical columns
            if not numerical_columns.empty:
                df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

            # Save the imputed data to a new CSV file in the output directory
            output_file = os.path.join(output_dir, f'{study_site}_imputed.csv')
            df.to_csv(output_file, index=False)
            print(f"Processed and saved imputed data for {study_site} to {output_file}")

        except Exception as e:
            print(f"Error processing file {filename}: {e}")


Dropped 'WGU' column from EARLWOOD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from EARLWOOD_AQMS_Processed.csv
Error processing file EARLWOOD_AQMS_Processed.csv: hour must be in 0..23: 01/01/2020 24:00
Dropped 'WGU' column from SINGLETON_AQMS_Processed.csv
Dropped columns with more than 50% missing values from SINGLETON_AQMS_Processed.csv
Error processing file SINGLETON_AQMS_Processed.csv: hour must be in 0..23: 01/01/2020 24:00
Dropped 'WGU' column from MAYFIELD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from MAYFIELD_AQMS_Processed.csv
Error processing file MAYFIELD_AQMS_Processed.csv: hour must be in 0..23: 01/01/2020 24:00
Dropped 'WGU' column from JERRYS-PLAINS_AQMS_Processed.csv
Dropped columns with more than 50% missing values from JERRYS-PLAINS_AQMS_Processed.csv
Error processing file JERRYS-PLAINS_AQMS_Processed.csv: hour must be in 0..23: 01/01/2020 24:00
Dropped 'WGU' column from MERRIWA_AQMS_Processed.csv
Dropped columns w

KNN Imputer

In [8]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Processed_Data'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Drop columns that have all missing values
            df = df.dropna(axis=1, how='all')

            # Sort the data by 'Time' column, keeping the 'Date' and 'Time' unchanged
            df = df.sort_values(by='Time')

            # Select only the numerical columns for imputation (excluding 'Date', 'Time', etc.)
            numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

            # Apply KNN imputation only to the numerical columns
            if not numerical_columns.empty:
                df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

            # Save the imputed data to a new CSV file in the output directory
            output_file = os.path.join(output_dir, f'{study_site}_imputed.csv')
            df.to_csv(output_file, index=False)
            print(f"Processed and saved imputed data for {study_site} to {output_file}")

        except Exception as e:
            print(f"Error processing file {filename}: {e}")

Processed and saved imputed data for RANDWICK to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/RANDWICK_imputed.csv
Processed and saved imputed data for EARLWOOD to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/EARLWOOD_imputed.csv
Processed and saved imputed data for ALEXANDRIA to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/ALEXANDRIA_imputed.csv
Processed and saved imputed data for CHULLORA to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/CHULLORA_imputed.csv
Processed and saved imputed data for LIDCOMBE2 to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/LIDCOMBE2_imputed.csv
Processed and saved imputed data for ROSELLE to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/ROSELLE_imputed.csv
Processed and saved imputed data for COOK&Philips to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Imputed_Data/COOK&Philips_imputed.csv
Processed and saved imputed data for LIDCOMBE to /mnt/scratch_lustre/barthelx/

Wind Vector Calculation:

In [3]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Function to ensure continuous days and 1-24 hours for each day
def ensure_continuous_days_and_hours(df, date_col, time_col):
    # Convert 'Date' to datetime and 'Time' to integer if not already in the correct format
    df[date_col] = pd.to_datetime(df[date_col])
    df[time_col] = df[time_col].astype(int)

    # Create a complete range of dates and hours (1-24 for each day)
    full_date_range = pd.date_range(start=df[date_col].min(), end=df[date_col].max(), freq='D')
    full_time_range = list(range(1, 25))  # Hours 1 to 24

    # Create a new DataFrame with all combinations of dates and hours
    full_datetime_index = pd.MultiIndex.from_product([full_date_range, full_time_range], names=[date_col, time_col])
    df = df.set_index([date_col, time_col]).reindex(full_datetime_index).reset_index()

    return df

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Drop the 'WGU' column if it exists
            if 'WGU' in df.columns:
                df = df.drop(columns=['WGU'])
                print(f"Dropped 'WGU' column from {filename}")

            # Drop columns that have more than 50% missing values
            missing_threshold = 0.5
            df = df.dropna(thresh=int(len(df) * (1 - missing_threshold)), axis=1)
            print(f"Dropped columns with more than 50% missing values from {filename}")

            # Drop columns that have all missing values
            df = df.dropna(axis=1, how='all')

            # Ensure continuous days and 1-24 hours for each day
            df = ensure_continuous_days_and_hours(df, date_col='Date', time_col='Time')

            # Select only the numerical columns for imputation (excluding 'Date', 'Time', etc.)
            numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

            # Apply KNN imputation only to the numerical columns
            if not numerical_columns.empty:
                df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

            # Save the imputed data to a new CSV file in the output directory
            output_file = os.path.join(output_dir, f'{study_site}_imputed.csv')
            df.to_csv(output_file, index=False)
            print(f"Processed and saved imputed data for {study_site} to {output_file}")

        except Exception as e:
            print(f"Error processing file {filename}: {e}")


Dropped 'WGU' column from EARLWOOD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from EARLWOOD_AQMS_Processed.csv
Error processing file EARLWOOD_AQMS_Processed.csv: invalid literal for int() with base 10: '01:00'
Dropped 'WGU' column from SINGLETON_AQMS_Processed.csv
Dropped columns with more than 50% missing values from SINGLETON_AQMS_Processed.csv
Error processing file SINGLETON_AQMS_Processed.csv: invalid literal for int() with base 10: '01:00'
Dropped 'WGU' column from MAYFIELD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from MAYFIELD_AQMS_Processed.csv
Error processing file MAYFIELD_AQMS_Processed.csv: invalid literal for int() with base 10: '01:00'
Dropped 'WGU' column from JERRYS-PLAINS_AQMS_Processed.csv
Dropped columns with more than 50% missing values from JERRYS-PLAINS_AQMS_Processed.csv
Error processing file JERRYS-PLAINS_AQMS_Processed.csv: invalid literal for int() with base 10: '01:00'
Dropped 'WGU' column from MERRIWA_AQMS

In [4]:
import os
import pandas as pd
from sklearn.impute import KNNImputer

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize the KNN Imputer
imputer = KNNImputer(n_neighbors=5)

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]
        
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Store the original index to preserve the row order
            df['original_order'] = range(len(df))

            # Drop the 'WGU' column if it exists
            if 'WGU' in df.columns:
                df = df.drop(columns=['WGU'])
                print(f"Dropped 'WGU' column from {filename}")

            # Drop columns that have more than 50% missing values
            missing_threshold = 0.5
            df = df.dropna(thresh=int(len(df) * (1 - missing_threshold)), axis=1)
            print(f"Dropped columns with more than 50% missing values from {filename}")

            # Drop columns that have all missing values
            df = df.dropna(axis=1, how='all')

            # Sort the data by 'Time' column temporarily for imputation purposes
            df_sorted = df.sort_values(by='Time')

            # Select only the numerical columns for imputation (excluding 'Date', 'Time', etc.)
            numerical_columns = df_sorted.select_dtypes(include=['float64', 'int64']).columns

            # Apply KNN imputation only to the numerical columns
            if not numerical_columns.empty:
                df_sorted[numerical_columns] = imputer.fit_transform(df_sorted[numerical_columns])

            # Return the data to its original order using 'original_order' column
            df = df_sorted.sort_values(by='original_order').drop(columns=['original_order'])

            # Save the imputed data to a new CSV file in the output directory
            output_file = os.path.join(output_dir, f'{study_site}_imputed.csv')
            df.to_csv(output_file, index=False)
            print(f"Processed and saved imputed data for {study_site} to {output_file}")

        except Exception as e:
            print(f"Error processing file {filename}: {e}")


Dropped 'WGU' column from EARLWOOD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from EARLWOOD_AQMS_Processed.csv
Processed and saved imputed data for EARLWOOD to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/EARLWOOD_imputed.csv
Dropped 'WGU' column from SINGLETON_AQMS_Processed.csv
Dropped columns with more than 50% missing values from SINGLETON_AQMS_Processed.csv
Processed and saved imputed data for SINGLETON to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/SINGLETON_imputed.csv
Dropped 'WGU' column from MAYFIELD_AQMS_Processed.csv
Dropped columns with more than 50% missing values from MAYFIELD_AQMS_Processed.csv
Processed and saved imputed data for MAYFIELD to /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Imputed_Data_NSW_ALL/MAYFIELD_imputed.csv
Dropped 'WGU' column from JERRYS-PLAINS_AQMS_Processed.csv
Dropped columns with more than 50% missing values from JERRYS-PLAINS_AQMS_Proce